# Lab 07

## Zadanie 1

Przygotuj demo programu znajdującego miejsca zerowe metodą Newtona. Wykorzystując `argparse` ([link](https://docs.python.org/3.6/library/argparse.html#module-argparse)) lub `optparse` ([link](https://docs.python.org/3.6/library/optparse.html)) obsłuż:
- ustalane punktu startowego,
- wielkość kroku w pochodnej,
- ilość kroków metody,
- dokładność
- pomoc

Program uruchamiamy podając, np.:

`./newton.py x**2+x+1 -h 0.00001`

In [1]:
# Zadanie 1 — demo programu newton.py
# -h oznacza krok pochodnej zgodnie z treścią zadania; pomoc jest pod --help.
from pathlib import Path
import subprocess
import sys

script = Path("newton.py")
if not script.exists():
    raise FileNotFoundError("Brak newton.py obok notebooka")

command = [
    sys.executable,
    str(script),
    "x**2 - 2",
    "--start",
    "1",
    "-h",
    "0.00001",
    "--iterations",
    "20",
    "--tolerance",
    "1e-12",
]
result = subprocess.run(command, text=True, capture_output=True, check=True)
print("$", " ".join(command))
print(result.stdout, end="")

# Przykład z polecenia: ./newton.py 'x**2+x+1' -h 0.00001
# (w liczbach rzeczywistych ten wielomian nie ma miejsca zerowego).

$ /Library/Frameworks/Python.framework/Versions/3.12/bin/python3 newton.py x**2 - 2 --start 1 -h 0.00001 --iterations 20 --tolerance 1e-12
root = 1.41421356237
f(root) = 4.44e-16
iterations = 5


## Zadanie 2

Zadanie należy wykonać wykorzystując program `BagOfWords` z poprzednich zajęć.


Przerób ją tak, żeby interpunkcja, cyfry i wszelkie inne znaki nie przeszkadzały w parsowaniu tekstu. Uruchom ja na [tekście hamleta](http://www.gutenberg.org/cache/epub/1787/pg1787.txt). Ile razy występuje słowo hamlet? Jak brzmi 10 najczęściej występujących słów?

In [2]:
# Zadanie 2 — BagOfWords odporny na interpunkcję i cyfry
from collections import Counter
from pathlib import Path
import re
from urllib.error import URLError
from urllib.request import Request, urlopen


WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*", re.UNICODE)


class BagOfWords:
    """Zlicza słowa bez rozróżniania wielkości liter."""

    def __init__(self, document=()):
        self._counts = Counter()
        chunks = (document,) if isinstance(document, str) else document
        for chunk in chunks:
            self._counts.update(word.lower() for word in WORD.findall(chunk))

    def __str__(self):
        return ", ".join(f"{word}:{self._counts[word]}" for word in self)

    __repr__ = __str__

    def __contains__(self, word):
        return word.lower() in self._counts

    def __iter__(self):
        return iter(sorted(self._counts, key=lambda word: (-self._counts[word], word)))

    def __add__(self, other):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords()
        result._counts = self._counts + other._counts
        return result

    def __getitem__(self, word):
        return self._counts[word.lower()]

    def __setitem__(self, word, count):
        if not isinstance(word, str) or not word:
            raise ValueError("słowo musi być niepustym napisem")
        if not isinstance(count, int) or isinstance(count, bool) or count < 0:
            raise ValueError("liczność musi być nieujemną liczbą całkowitą")
        if count:
            self._counts[word.lower()] = count
        else:
            self._counts.pop(word.lower(), None)


# Sprawdzenie normalizacji.
sample = BagOfWords("Hamlet, HAMLET! 123 hamlet's.")
assert sample["hamlet"] == 2 and sample["hamlet's"] == 1

HAMLET_URL = "https://www.gutenberg.org/cache/epub/1787/pg1787.txt"
LOCAL_HAMLET = Path("hamlet.txt")


def gutenberg_body(text):
    """Usuń nagłówek i stopkę Project Gutenberg, jeśli występują."""
    start = re.search(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    if start:
        text = text[start.end():]
    end = re.search(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    if end:
        text = text[:end.start()]
    return text


def load_hamlet():
    """Czytaj lokalny plik albo pobierz tekst; nie zapisuj automatycznie pobrania."""
    if LOCAL_HAMLET.exists():
        return LOCAL_HAMLET.read_text(encoding="utf-8")
    request = Request(HAMLET_URL, headers={"User-Agent": "BagOfWords lab exercise"})
    with urlopen(request, timeout=20) as response:
        return response.read().decode("utf-8")


try:
    hamlet = BagOfWords(gutenberg_body(load_hamlet()))
except (OSError, URLError, UnicodeDecodeError) as error:
    hamlet = None
    print(f"Blokada zewnętrzna: brak hamlet.txt i pobranie z Project Gutenberg nie powiodło się: {error}")
else:
    print(f"hamlet: {hamlet['hamlet']}")
    print("10 najczęstszych słów:", [(word, hamlet[word]) for word in list(hamlet)[:10]])

hamlet: 111
10 najczęstszych słów: [('the', 1163), ('and', 1040), ('to', 798), ('of', 765), ('a', 587), ('you', 587), ('i', 575), ('my', 520), ('in', 443), ('it', 431)]


## Zadanie 3

Wykorzystując `pickle` zapisz i odczytaj klasy z poprzedniego zadania nakarmonej Hamletem. Porównaj metody i rozmiar.

In [3]:
# Zadanie 3 — pickle
# Uruchom po zadaniu 2; pickle zachowuje obiekt BagOfWords wraz z Counterem.
import io
import pickle

if hamlet is None:
    print("Blokada zewnętrzna: tekst Hamleta niedostępny; porównanie pickle pominięte.")
else:
    # Oba strumienie powstają tutaj, więc ich odczyt jest bezpieczny.
    payload = pickle.dumps(hamlet, protocol=pickle.HIGHEST_PROTOCOL)
    restored_from_bytes = pickle.loads(payload)

    binary = io.BytesIO()
    pickle.dump(hamlet, binary, protocol=pickle.HIGHEST_PROTOCOL)
    stream_size = binary.tell()
    binary.seek(0)
    restored_from_stream = pickle.load(binary)

    assert isinstance(restored_from_bytes, BagOfWords)
    assert restored_from_bytes._counts == hamlet._counts
    assert restored_from_stream._counts == hamlet._counts
    print(f"dumps/loads: {len(payload):,} B")
    print(f"dump/load przez plik: {stream_size:,} B")
    print("Rozmiary są równe: BytesIO nie dodaje narzutu; nie odczytuj pickle z niezaufanych źródeł.")

dumps/loads: 57,991 B
dump/load przez plik: 57,991 B
Rozmiary są równe: BytesIO nie dodaje narzutu; nie odczytuj pickle z niezaufanych źródeł.


## Zadanie 4

Wykorzystując [https://gist.github.com/pamelafox/986163](https://gist.github.com/pamelafox/986163) podaj aktualną godzinę we:
- wszystkich krajach, wyświetlając je zgrupowane względem kontynentów,
- przeprowadź symulacyjne wyświetlanie kolejnych krajów w miarę jak w danym kraju wybija północ, opóżnienie wyświetlania ustaw proporcjonalne do realnego czasu

In [4]:
# Zadanie 4 — strefy czasowe
# Dane krajów/stref pochodzą z bazy pytz (odpowiednik listy z podanego gist).
from collections import defaultdict
from datetime import datetime, timedelta
from time import sleep

try:
    import pytz
except ImportError as error:
    raise RuntimeError("Brak pytz. Zainstaluj je: pip install pytz") from error


CONTINENTS = ("Africa", "America", "Antarctica", "Asia", "Atlantic", "Australia", "Europe", "Indian", "Pacific")


def continent(timezone_name):
    """Zwróć kontynent wywnioskowany z nazwy strefy IANA."""
    first_part = timezone_name.split("/", 1)[0]
    return first_part if first_part in CONTINENTS else "Other"


def country_times(now=None):
    """Zgrupuj kraje i ich lokalne czasy według kontynentu."""
    now = now or datetime.now(pytz.UTC)
    grouped = defaultdict(list)
    for code, zones in pytz.country_timezones.items():
        local_times = [(zone, now.astimezone(pytz.timezone(zone))) for zone in zones]
        group = continent(zones[0]) if zones else "Other"
        grouped[group].append((pytz.country_names.get(code, code), code, local_times))
    return {group: sorted(countries) for group, countries in sorted(grouped.items())}


def print_country_times(now=None):
    for group, countries in country_times(now).items():
        print(f"\\n{group}")
        for name, code, local_times in countries:
            values = ", ".join(f"{zone}: {time:%Y-%m-%d %H:%M}" for zone, time in local_times)
            print(f"  {name} ({code}) — {values}")


def next_midnight(local_time):
    tomorrow = local_time.date() + timedelta(days=1)
    return local_time.tzinfo.localize(datetime.combine(tomorrow, datetime.min.time()))


def midnight_events(now=None):
    """Zwróć kolejne kraje wraz z najbliższą lokalną północą."""
    now = now or datetime.now(pytz.UTC)
    events = []
    for name, code, local_times in sum((countries for countries in country_times(now).values()), []):
        midnight = min(next_midnight(local_time) for _, local_time in local_times)
        events.append((midnight.astimezone(pytz.UTC), name, code, midnight))
    return iter(sorted(events))


def simulate_midnights(limit=5, real_seconds_per_display_second=3600, now=None):
    """Wyświetl kraje przy północy; 1 s programu = podana liczba sekund realnych."""
    if limit < 1 or real_seconds_per_display_second <= 0:
        raise ValueError("limit i real_seconds_per_display_second muszą być dodatnie")

    previous = None
    for moment_utc, name, code, local_midnight in list(midnight_events(now))[:limit]:
        if previous is not None:
            delay = (moment_utc - previous).total_seconds() / real_seconds_per_display_second
            sleep(delay)
        print(f"Północ: {name} ({code}), {local_midnight.tzname()}")
        previous = moment_utc


# Pełna lista jest długa — odkomentuj dla wszystkich krajów:
# print_country_times()
# Symulacja pięciu kolejnych północy, przyspieszona 3600 razy:
# simulate_midnights(limit=5, real_seconds_per_display_second=3600)
print(f"Krajów w bazie: {len(pytz.country_timezones)}")

Krajów w bazie: 247


## Zadanie 5

Dla klasy `BagOfWords` napisz metody `save` oraz `load` wykorzystujące `json`'a do zapisu i odczytu danych.

In [5]:
# Zadanie 5 — zapis JSON
# Uruchom po komórce z zadania 2.
import json
from collections import Counter
from pathlib import Path


def _save(self, path):
    """Zapisz częstości słów do pliku JSON."""
    Path(path).write_text(
        json.dumps({"counts": dict(self._counts)}, ensure_ascii=False, sort_keys=True),
        encoding="utf-8",
    )


@classmethod
def _load(cls, path):
    """Odczytaj poprawny plik utworzony przez ``save``."""
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        counts = data["counts"]
    except (OSError, json.JSONDecodeError, KeyError, TypeError) as error:
        raise ValueError(f"niepoprawny plik BagOfWords: {path}") from error

    if not isinstance(counts, dict) or any(
        not isinstance(word, str)
        or not isinstance(count, int)
        or isinstance(count, bool)
        or count < 0
        for word, count in counts.items()
    ):
        raise ValueError("JSON musi zawierać słownik string -> nieujemna liczba całkowita")

    result = cls(())
    result._counts = Counter({word: count for word, count in counts.items() if count})
    return result


BagOfWords.save = _save
BagOfWords.load = _load

# Test bez zostawiania pliku w repozytorium.
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "bag.json"
    bow = BagOfWords("Ala, ala! kot 123 kot.")
    bow.save(path)
    restored = BagOfWords.load(path)
    assert restored._counts == bow._counts

print("save/load JSON: OK")

save/load JSON: OK
